# WorldSim v2 Training Data Generation

Generate Tasks I–N + Task G Korean fix via Sonnet API (OpenRouter).


## 1. Repo Root & Environment


In [1]:
from pathlib import Path
import sys, os

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "config" / "generation.yaml").exists():
    REPO_ROOT = REPO_ROOT.parent
    if not (REPO_ROOT / "config" / "generation.yaml").exists():
        raise RuntimeError("Cannot find repo root. Run this notebook from the worldsim-training directory.")

sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

from scripts.generate_data import load_local_env
load_local_env(REPO_ROOT)

api_key = os.environ.get("OPENROUTER_API_KEY", "")
model = os.environ.get("OPENROUTER_MODEL", "")
print(f"Repo root: {REPO_ROOT}")
print(f"API key:   {'✅ set (' + api_key[:12] + '...)' if api_key else '❌ MISSING — set OPENROUTER_API_KEY in .env'}")
print(f"Model:     {model or '⚠️ using default from generation.yaml'}")
assert api_key, "OPENROUTER_API_KEY is required. Add it to .env file."


Repo root: /home/hyunlord/github/worldsim-training
API key:   ✅ set (sk-or-v1-7ce...)
Model:     anthropic/claude-sonnet-4.6


## 2. Batch Preview (Dry Run)


In [2]:
from collections import Counter

from scripts.generate_data import (
    apply_batch_plan_to_settings,
    batch_task_counts,
    build_jobs,
    load_batch_plan,
    load_generation_config,
    select_requested_jobs,
)

for batch_id in ["batch_v2_01_tasks_in", "batch_v2_02_task_g_fix"]:
    batch_plan = load_batch_plan(REPO_ROOT, batch_id=batch_id)
    settings = apply_batch_plan_to_settings(load_generation_config(REPO_ROOT / "config"), batch_plan)
    jobs = build_jobs(REPO_ROOT, settings_override=settings)
    selected = select_requested_jobs(jobs, task_counts=batch_task_counts(batch_plan))
    counts = Counter(j["task"] for j in selected)
    print(f"\n{'=' * 40}")
    print(f"Batch: {batch_id}")
    print(f"Total jobs: {len(selected)}")
    for task, count in sorted(counts.items()):
        print(f"  Task {task}: {count}")
    print(f"{'=' * 40}")



Batch: batch_v2_01_tasks_in
Total jobs: 1100
  Task I: 200
  Task J: 200
  Task K: 200
  Task L: 200
  Task M: 150
  Task N: 150

Batch: batch_v2_02_task_g_fix
Total jobs: 200
  Task G: 200


## 3. Batch 1: Tasks I–N Generation


In [4]:
import time

from scripts.generate_data import generate_dataset, load_batch_plan

batch_plan_1 = load_batch_plan(REPO_ROOT, batch_id="batch_v2_01_tasks_in")
print(f"Starting Batch 1: Tasks I–N ({sum(batch_plan_1.get('task_counts', {}).values())} examples)")
print("Estimated cost: ~$4.55")
print("Estimated time: 30-60 minutes")
print("=" * 50)

started = time.perf_counter()
result_1 = generate_dataset(REPO_ROOT, batch_plan=batch_plan_1)
elapsed_1 = time.perf_counter() - started

print(f"\n{'=' * 50}")
print(f"Batch 1 complete in {elapsed_1:.0f}s ({elapsed_1 / 60:.1f}min)")
result_1


Starting Batch 1: Tasks I–N (1100 examples)
Estimated cost: ~$4.55
Estimated time: 30-60 minutes
[retry 1/2] task=I variant=0 validation=too_long
[retry 2/2] task=I variant=0 validation=forbidden_in_reasoning_ko,too_long
[skip] task=I variant=0 reason=generation_validation_failed:too_long
[retry 1/2] task=K variant=0 validation=forbidden_in_hint_ko,too_long
[retry 2/2] task=K variant=0 validation=forbidden_in_hint_ko,too_long
[skip] task=K variant=0 reason=generation_validation_failed:too_long
[retry 1/2] task=J variant=1 validation=too_long
[retry 1/2] task=L variant=1 validation=too_long
[retry 2/2] task=L variant=1 validation=too_long
[10/1100] task=L success=8 skipped=2 success_rate=80.0% elapsed=93.46s last=13.93s rows_per_second=0.11 eta_seconds=10187.42 tokens=23455 estimated_cost_usd=$0.092733
  prompt_tokens=21591 completion_tokens=1864 tokens_per_second=250.96
  by_task_success=I:1, J:2, K:1, L:2, M:1, N:1 by_task_skipped=I:1, K:1
  dominant_failures=I:generation_validation_f

{'batch_id': 'batch_v2_01_tasks_in',
 'output_path': PosixPath('/home/hyunlord/github/worldsim-training/data/raw/batch_v2_01_tasks_in/generated.jsonl'),
 'skipped_path': PosixPath('/home/hyunlord/github/worldsim-training/data/raw/batch_v2_01_tasks_in/skipped.jsonl'),
 'progress_path': PosixPath('/home/hyunlord/github/worldsim-training/data/raw/batch_v2_01_tasks_in/progress.json'),
 'summary_path': PosixPath('/home/hyunlord/github/worldsim-training/data/raw/batch_v2_01_tasks_in/summary.json'),
 'log_path': PosixPath('/home/hyunlord/github/worldsim-training/data/raw/batch_v2_01_tasks_in/generate.log'),
 'count': 1031,
 'record_count': 1031,
 'jobs_count': 1100,
 'skipped_count': 69,
 'prompt_tokens': 1837795,
 'completion_tokens': 157838,
 'total_tokens': 1995633,
 'estimated_cost_usd': 7.880955,
 'elapsed_seconds': 6398.22621218604,
 'rows_per_second': 0.17192264910936467,
 'tokens_per_second': 311.90410182733524,
 'task_counts': {'J': 185, 'L': 191, 'M': 148, 'N': 143, 'I': 176, 'K': 1

## 4. Batch 1 Summary


In [5]:
import json

batch1_dir = REPO_ROOT / "data" / "raw" / "batch_v2_01_tasks_in"
summary_path = batch1_dir / "summary.json"
generated_path = batch1_dir / "generated.jsonl"
skipped_path = batch1_dir / "skipped.jsonl"

if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    print("=== Batch 1 Summary ===")
    for key in ["total_generated", "total_skipped", "total_cost_usd", "elapsed_seconds"]:
        print(f"  {key}: {summary.get(key, 'N/A')}")
    print("\n  Task breakdown:")
    for task, count in sorted(summary.get("task_counts", {}).items()):
        print(f"    Task {task}: {count}")

generated_count = sum(1 for _ in open(generated_path, encoding="utf-8")) if generated_path.exists() else 0
skipped_count = sum(1 for _ in open(skipped_path, encoding="utf-8")) if skipped_path.exists() else 0
print(f"\n  Generated rows: {generated_count}")
print(f"  Skipped rows:  {skipped_count}")

if generated_path.exists():
    print("\n=== Sample Outputs ===")
    with open(generated_path, encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= 3:
                break
            row = json.loads(line)
            print(f"\n--- Task {row.get('task')} ---")
            output = row.get("output", row.get("assistant", ""))
            if isinstance(output, str) and len(output) > 200:
                output = output[:200] + "..."
            print(f"  Output: {output}")


=== Batch 1 Summary ===
  total_generated: N/A
  total_skipped: N/A
  total_cost_usd: N/A
  elapsed_seconds: N/A

  Task breakdown:

  Generated rows: 1031
  Skipped rows:  69

=== Sample Outputs ===

--- Task J ---
  Output: {"coping_id":3,"coping_type":"ritualistic","stress_delta":-0.3,"hint_ko":"규율을 지키며 제삿불 곁에서 마음을 추스른다","hint_en":"She steadies her shattered heart by faithfully tending the memorial flame according to st...

--- Task L ---
  Output: {"response_id":1,"trust_delta":0.08,"hint_ko":"함께 목숨 걸었으니 몫만큼은 제대로 따져야 안심된다","hint_en":"Having risked their lives together, carefully verifying each share brings a sense of security.","forgiveness_thr...

--- Task M ---
  Output: {"decision_id":0,"confidence":0.78,"dissent_risk":0.42,"reasoning_ko":"굶주림을 당장 끊으려면 사냥대를 불려 먹거리를 채워야 한다","reasoning_en":"To cut off the spreading hunger immediately, the group must expand the hunting ...


## 5. Batch 2: Task G Korean Fix


In [6]:
batch_plan_2 = load_batch_plan(REPO_ROOT, batch_id="batch_v2_02_task_g_fix")
print(f"Starting Batch 2: Task G Korean fix ({sum(batch_plan_2.get('task_counts', {}).values())} examples)")
print("Estimated cost: ~$0.90")
print("=" * 50)

started = time.perf_counter()
result_2 = generate_dataset(REPO_ROOT, batch_plan=batch_plan_2)
elapsed_2 = time.perf_counter() - started

print(f"\n{'=' * 50}")
print(f"Batch 2 complete in {elapsed_2:.0f}s ({elapsed_2 / 60:.1f}min)")
result_2


Starting Batch 2: Task G Korean fix (200 examples)
Estimated cost: ~$0.90
[retry 1/2] task=G variant=0 validation=too_long
[retry 2/2] task=G variant=0 validation=too_long
[retry 1/2] task=G variant=0 validation=too_long
[retry 2/2] task=G variant=0 validation=too_long
[skip] task=G variant=0 reason=generation_validation_failed:too_long
[retry 1/2] task=G variant=0 validation=too_long
[retry 2/2] task=G variant=0 validation=too_long
[skip] task=G variant=0 reason=generation_validation_failed:too_long
[retry 1/2] task=G variant=0 validation=too_long
[retry 2/2] task=G variant=0 validation=too_long
[skip] task=G variant=0 reason=generation_validation_failed:too_long
[retry 1/2] task=G variant=0 validation=too_long
[retry 1/2] task=G variant=0 validation=register_mismatch
[retry 1/2] task=G variant=0 validation=too_long
[10/200] task=G success=7 skipped=3 success_rate=70.0% elapsed=105.04s last=5.70s rows_per_second=0.10 eta_seconds=1995.83 tokens=34420 estimated_cost_usd=$0.137484
  prom

{'batch_id': 'batch_v2_02_task_g_fix',
 'output_path': PosixPath('/home/hyunlord/github/worldsim-training/data/raw/batch_v2_02_task_g_fix/generated.jsonl'),
 'skipped_path': PosixPath('/home/hyunlord/github/worldsim-training/data/raw/batch_v2_02_task_g_fix/skipped.jsonl'),
 'progress_path': PosixPath('/home/hyunlord/github/worldsim-training/data/raw/batch_v2_02_task_g_fix/progress.json'),
 'summary_path': PosixPath('/home/hyunlord/github/worldsim-training/data/raw/batch_v2_02_task_g_fix/summary.json'),
 'log_path': PosixPath('/home/hyunlord/github/worldsim-training/data/raw/batch_v2_02_task_g_fix/generate.log'),
 'count': 154,
 'record_count': 154,
 'jobs_count': 200,
 'skipped_count': 46,
 'prompt_tokens': 523900,
 'completion_tokens': 45926,
 'total_tokens': 569826,
 'estimated_cost_usd': 2.26059,
 'elapsed_seconds': 1665.6751804861706,
 'rows_per_second': 0.12007142949781169,
 'tokens_per_second': 342.09911192510026,
 'task_counts': {'G': 154},
 'planned_task_counts': {'G': 200},
 '

## 6. Batch 2 Summary


In [7]:
batch2_dir = REPO_ROOT / "data" / "raw" / "batch_v2_02_task_g_fix"
summary_path_2 = batch2_dir / "summary.json"
generated_path_2 = batch2_dir / "generated.jsonl"

if summary_path_2.exists():
    summary_2 = json.loads(summary_path_2.read_text(encoding="utf-8"))
    print("=== Batch 2 Summary ===")
    for key in ["total_generated", "total_skipped", "total_cost_usd", "elapsed_seconds"]:
        print(f"  {key}: {summary_2.get(key, 'N/A')}")

generated_count_2 = sum(1 for _ in open(generated_path_2, encoding="utf-8")) if generated_path_2.exists() else 0
print(f"\n  Generated rows: {generated_count_2}")

if generated_path_2.exists():
    print("\n=== Task G Sample Outputs (check Korean quality) ===")
    with open(generated_path_2, encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= 3:
                break
            row = json.loads(line)
            output = row.get("output", row.get("assistant", ""))
            print(f"\n--- G sample {i + 1} ---")
            if isinstance(output, str):
                try:
                    parsed = json.loads(output)
                    print(f"  interpretation_ko: {parsed.get('interpretation_ko', 'N/A')}")
                    print(f"  interpretation_en: {parsed.get('interpretation_en', 'N/A')}")
                except json.JSONDecodeError:
                    print(f"  Raw: {output[:200]}")


=== Batch 2 Summary ===
  total_generated: N/A
  total_skipped: N/A
  total_cost_usd: N/A
  elapsed_seconds: N/A

  Generated rows: 154

=== Task G Sample Outputs (check Korean quality) ===

--- G sample 1 ---
  interpretation_ko: 북쪽 산 너머에 정말 풍요로운 것이 있는지 먼저 꼼꼼히 살펴보아야 하오.
  interpretation_en: We must carefully verify whether true abundance truly lies beyond the northern mountains before acting.

--- G sample 2 ---
  interpretation_ko: 북쪽 산 너머에 넘쳐나는 먹거리가 기다리고 있어, 당장 옮겨가야 해.
  interpretation_en: Abundant provisions lie waiting beyond the northern mountains, so we must move there at once.

--- G sample 3 ---
  interpretation_ko: 북쪽 산만 넘으면 풍족해진다는 거잖아, 지금 당장 떠나야 해
  interpretation_en: It's saying there's plenty just beyond the northern mountains, so we have to leave right now.


## 7. Negative Examples & General Korean Stats


In [8]:
from collections import Counter

neg_path = REPO_ROOT / "data" / "samples" / "negative_examples.jsonl"
gen_path = REPO_ROOT / "data" / "samples" / "general_korean.jsonl"

for label, path in [("Negative Examples", neg_path), ("General Korean", gen_path)]:
    if path.exists():
        with open(path, encoding="utf-8") as f:
            rows = [json.loads(line) for line in f if line.strip()]
        cats = Counter(r.get("reason", r.get("category", "unknown")) for r in rows)
        print(f"\n=== {label}: {len(rows)} total ===")
        for cat, count in sorted(cats.items(), key=lambda x: -x[1]):
            print(f"  {cat}: {count}")
    else:
        print(f"\n⚠️ {label}: file not found at {path}")



=== Negative Examples: 500 total ===
  repetition_loop: 100
  forbidden_sino_korean: 100
  register_mixing: 80
  modern_vocabulary: 50
  excessive_length: 50
  schema_leakage: 50
  key_hallucination: 40
  english_in_ko_field: 30

=== General Korean: 300 total ===
  qa: 100
  creative: 80
  json_formatting: 60
  instruction: 30
  proverb: 30


## 8. Validation


In [9]:
from scripts.common import load_yaml, resolve_path
from scripts.validate_data import _load_batch_plan as load_validation_batch_plan
from scripts.validate_data import _resolve_batch_paths, load_validation_rules, validate_file

print("=== Validating Batch 1 (Tasks I–N) ===")
batch1_generated = REPO_ROOT / "data" / "raw" / "batch_v2_01_tasks_in" / "generated.jsonl"
if batch1_generated.exists():
    settings = load_yaml(REPO_ROOT / "config" / "generation.yaml")
    batch_plan = load_validation_batch_plan(REPO_ROOT, "batch_v2_01_tasks_in")
    input_path, output_dir = _resolve_batch_paths(REPO_ROOT, settings, batch_plan)
    summary = validate_file(input_path=input_path, validated_dir=output_dir, rules=load_validation_rules(REPO_ROOT / "config"))
    print(summary)
else:
    print("  ⚠️ Batch 1 not yet generated")

print("\n=== Validating Batch 2 (Task G fix) ===")
batch2_generated = REPO_ROOT / "data" / "raw" / "batch_v2_02_task_g_fix" / "generated.jsonl"
if batch2_generated.exists():
    settings = load_yaml(REPO_ROOT / "config" / "generation.yaml")
    batch_plan = load_validation_batch_plan(REPO_ROOT, "batch_v2_02_task_g_fix")
    input_path, output_dir = _resolve_batch_paths(REPO_ROOT, settings, batch_plan)
    summary = validate_file(input_path=input_path, validated_dir=output_dir, rules=load_validation_rules(REPO_ROOT / "config"))
    print(summary)
else:
    print("  ⚠️ Batch 2 not yet generated")


=== Validating Batch 1 (Tasks I–N) ===
{'input_file': '/home/hyunlord/github/worldsim-training/data/raw/batch_v2_01_tasks_in/generated.jsonl', 'passed': 1031, 'failed': 0, 'total': 1031, 'violations': {}}

=== Validating Batch 2 (Task G fix) ===
{'input_file': '/home/hyunlord/github/worldsim-training/data/raw/batch_v2_02_task_g_fix/generated.jsonl', 'passed': 154, 'failed': 0, 'total': 154, 'violations': {}}


## 9. Overall v2 Data Status


In [10]:
print("=" * 60)
print("V2 DATA GENERATION STATUS")
print("=" * 60)

sources = {
    "v1 train": REPO_ROOT / "data" / "training" / "worldsim-v31-mix-v1" / "train_converted.jsonl",
    "v1 dev": REPO_ROOT / "data" / "training" / "worldsim-v31-mix-v1" / "dev_converted.jsonl",
    "Batch 1 (I-N)": REPO_ROOT / "data" / "raw" / "batch_v2_01_tasks_in" / "generated.jsonl",
    "Batch 2 (G fix)": REPO_ROOT / "data" / "raw" / "batch_v2_02_task_g_fix" / "generated.jsonl",
    "Negative": REPO_ROOT / "data" / "samples" / "negative_examples.jsonl",
    "General KR": REPO_ROOT / "data" / "samples" / "general_korean.jsonl",
}

grand_total = 0
for label, path in sources.items():
    if path.exists():
        count = sum(1 for line in open(path, encoding="utf-8") if line.strip())
        print(f"  {label:20s}: {count:>5} rows")
        grand_total += count
    else:
        print(f"  {label:20s}:   N/A (not yet generated)")

print(f"\n  {'GRAND TOTAL':20s}: {grand_total:>5} rows")
print(f"\n  Estimated API cost: ~${4.55 + 0.90:.2f}")
print("=" * 60)


V2 DATA GENERATION STATUS
  v1 train            :  2201 rows
  v1 dev              :   251 rows
  Batch 1 (I-N)       :  1031 rows
  Batch 2 (G fix)     :   154 rows
  Negative            :   500 rows
  General KR          :   300 rows

  GRAND TOTAL         :  4437 rows

  Estimated API cost: ~$5.45
